### Initial Data Audit

#### Purpose

Inspect the Olist dataset tables to exclude unneeded tables and columns to guide the construction of an Analytical Base Table (ABT) with one row per order.

The project aims to predict delivery delays using information available
at order approval.

#### Scope

For each table:
- Identify its granularity, keys, and relationships.
- Check data quality issues relevant to ABT construction.
- Assess which fields support the prediction task.
- Document findings and decisions about joins, aggregation, and column selection.

In [2]:
import pandas as pd

#### customers 

The table has information about the recipient's location. We can use customer_id for the join to build the ABT and use customer_zip_code_prefix to extract the location and make calculations. The features customer_city and customer_state can be used as categorical features. customer_unique_id will be excluded. customer_zip_code_prefix will be read as text to preserve leading zeros.

In [7]:
df_costumers = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "str"}
)
print(f"{df_costumers.info()}\n")

df_costumers.head()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB
None



,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


#### geolocation

The table will provide coordinates for customer and seller locations, retain the ZIP code prefix for joins and latitude and longitude for straight-line distance calculations. 

Exclude geolocation_city and geolocation_state because city and state will be retained in the customers and sellers tables. geolocation_zip_code_prefix will be read as text to preserve leading zeros.

In [3]:
df_geol = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
df_geol.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


#### order_items

The table connects orders to their products and sellers and provides item-level
prices, freight values, and shipping deadlines. Item-level information
will need to be aggregated to build an
ABT with one row per order.

Drop shipping_limit_date during selection because its availability at
order approval could not be verified.

In [ ]:
df_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
df_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


#### payments

This table provides payment methods, installment counts, and payment values
for each order. order_id supports the join, while payment_sequential identifies payment records within an order.

I decided to exclude payments at data selection. The model predicts delivery delay at order approval, when payment approval has already occurred. I expect delivery conditions and order contents to be more useful for this problem than payment method, installments, or payment value. The raw table is still useful for checking item prices and freight against recorded payments.

In [4]:
df_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
df_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


#### reviews

The table contains customer feedback collected after order approval. It will be excluded from the ABT because its information is unavailable at prediction time and is not required to define the delivery delay target.

In [9]:
df_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
df_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


#### orders

The table will serve as the starting point for the ABT, with order_id
identifying each order and customer_id supporting the join to customers. 

order_purchase_timestamp, order_approved_at, and
order_estimated_delivery_date will be retained as information potentially
available at prediction time. order_delivered_customer_date will be used
with the estimated delivery date to define the delay target.

order_status will be retained for population selection, but excluded from
model features. order_delivered_carrier_date will be excluded from the
initial ABT because it occurs after approval and is not needed to define
the target.

The initial analysis will focus on delivered orders. Among the 96,478
orders with this status, 8 lack the customer delivery timestamp and 14
lack the approval timestamp. These groups may overlap; handling these
records will be addressed during data preparation.

In [22]:
df_orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
df_orders[df_orders["order_status"] == "delivered"].info()

<class 'pandas.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       96478 non-null  str  
 1   customer_id                    96478 non-null  str  
 2   order_status                   96478 non-null  str  
 3   order_purchase_timestamp       96478 non-null  str  
 4   order_approved_at              96464 non-null  str  
 5   order_delivered_carrier_date   96476 non-null  str  
 6   order_delivered_customer_date  96470 non-null  str  
 7   order_estimated_delivery_date  96478 non-null  str  
dtypes: str(8)
memory usage: 6.6 MB


#### products

The table provides product categories and physical characteristics.
product_id will support the join to order_items. Product category,
weight, and dimensions will be retained as potential model features.

product_name_lenght, product_description_lenght, and product_photos_qty
will be excluded from the initial scope because they describe the listing
rather than the physical or logistical characteristics of the product.

In [11]:
df_products = pd.read_csv("../data/raw/olist_products_dataset.csv")
df_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

df_products = df_products.merge(
    df_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

df_products = df_products.drop(
    columns="product_category_name"
).rename(
    columns={"product_category_name_english": "product_category_name"}
)

df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_name_lenght         32341 non-null  float64
 2   product_description_lenght  32341 non-null  float64
 3   product_photos_qty          32341 non-null  float64
 4   product_weight_g            32949 non-null  float64
 5   product_length_cm           32949 non-null  float64
 6   product_height_cm           32949 non-null  float64
 7   product_width_cm            32949 non-null  float64
 8   product_category_name       32328 non-null  str    
dtypes: float64(7), str(2)
memory usage: 2.3 MB


#### sellers

The table provides the seller's location. seller_id will support the join to order_items, and seller_zip_code_prefix will connect sellers to geolocation to obtain origin coordinates and estimate distances to recipients. 

The features seller_city and seller_state can be used as categorical features. seller_zip_code_prefix will be read as text to preserve leading zeros.

In [6]:
df_sellers = pd.read_csv(
    "../data/raw/olist_sellers_dataset.csv",
    dtype={"seller_zip_code_prefix": "str"}
)
df_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
